# DifferentialNumpy — Benchmark Suite

**Branch:** `v3`  |  **Target:** CPU & CUDA  
**Paradigm:** clone → run cell-by-cell → fix locally → commit & push → restart kernel → re-run

---

### What this notebook covers

| # | Section | What is measured |
|---|---------|-----------------|
| 1 | **Environment setup** | Colab clone, pip install, device detection |
| 2 | **Op microbenchmarks** | Per-op forward timing: element-wise, reductions, activations, conv/pool |
| 3 | **Device tracking** | `device_op_percentage` — % of ops on CPU vs CUDA |
| 4 | **Model benchmarks** | MLP & CNN forward+backward latency, batch-size scaling |
| 5 | **Training benchmark** | Full training loop throughput (samples/s), CPU vs CUDA comparison |
| 6 | **GPU memory** | CUDA memory footprint per model |
| 7 | **Summary** | Combined results table |


In [ ]:
# ── 0. Colab / local setup ────────────────────────────────────────────────────
import sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/DifferentialNumpy" if IN_COLAB else os.path.abspath(
    os.path.join(os.getcwd(), "..")
)

if IN_COLAB:
    import subprocess
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", "v3",
             "https://github.com/Seydifa/AutoDiff-Numpy.git", REPO_DIR],
            check=True,
        )
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"], check=True)
    # Optional GPU backend
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "cupy-cuda12x", "--quiet"], check=True)
    except Exception:
        print("cupy not installed — CUDA benchmarks will be skipped.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo dir:", REPO_DIR)
print("Python  :", sys.version.split()[0])


In [ ]:
# ── 1. Imports & device detection ─────────────────────────────────────────────
import time, warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

import dnp
import dnp.ops as ops
from dnp.core.backend import backend, is_cuda_available, synchronize, get_device_count
from dnp.utils import device_op_percentage, reset_device_stats

# ── Detect CUDA ───────────────────────────────────────────────────────────────
CUDA_OK = is_cuda_available
DEVICES  = ["cpu"] + (["cuda"] if CUDA_OK else [])

print(f"{'='*55}")
print(f"  NumPy version  : {np.__version__}")
print(f"  CUDA available : {CUDA_OK}")
if CUDA_OK:
    import cupy as cp
    print(f"  CuPy  version  : {cp.__version__}")
    print(f"  GPU count      : {get_device_count()}")
    dev = cp.cuda.Device(0)
    print(f"  GPU 0          : {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
print(f"{'='*55}")


---
## Section 1 — Op Microbenchmarks

Measure per-operation **forward-pass latency (ms)** with and without autograd overhead.

- **`fwd_only`** — raw backend call (NumPy/CuPy), no graph recording  
- **`fwd_graph`** — call through `Ops.__call__` (Tensor node creation + DAG wiring)  
- **`overhead`** — `fwd_graph − fwd_only` (pure autograd bookkeeping cost)  
- **`bwd`** — full forward + `backward()` to scalar loss


In [ ]:
# ── Timing helpers ─────────────────────────────────────────────────────────────
REPS_FWD = 200   # repetitions for forward-only timing
REPS_BWD = 50    # repetitions for backward timing

def _sync():
    """GPU synchronize before stopping the clock (no-op on CPU)."""
    synchronize()

def time_raw(fn, *args, reps=REPS_FWD):
    """Time a raw backend call (no autograd). Returns mean ms."""
    _sync()
    t0 = time.perf_counter()
    for _ in range(reps):
        fn(*args)
        _sync()
    return (time.perf_counter() - t0) * 1000 / reps

def time_fwd(op, *tensors, reps=REPS_FWD):
    """Time an Ops forward call (autograd overhead included). Returns mean ms."""
    _sync()
    t0 = time.perf_counter()
    for _ in range(reps):
        op(*tensors)
        _sync()
    return (time.perf_counter() - t0) * 1000 / reps

def time_bwd(op, *tensors, reps=REPS_BWD):
    """Time forward + backward pass. Returns mean ms."""
    from dnp.core.session import session
    _sync()
    t0 = time.perf_counter()
    for _ in range(reps):
        session.reset()
        out = op(*tensors)
        # reduce to scalar if needed
        if hasattr(out, "data") and out.data.ndim > 0:
            loss = ops.mean(ops.sum(out))
        else:
            loss = out
        if hasattr(loss, "backward"):
            loss.backward()
        _sync()
    return (time.perf_counter() - t0) * 1000 / reps

def make_tensor(shape, device="cpu"):
    """Create a random Tensor on the specified device."""
    dnp.set_device(device)
    return dnp.Tensor(np.random.randn(*shape).astype(np.float32))

def benchmark_op(name, raw_fn, op, shapes, device="cpu", reps_fwd=REPS_FWD, reps_bwd=REPS_BWD):
    """Run full benchmark for one op on one device. Returns a dict row."""
    dnp.set_device(device)
    tensors = [make_tensor(s, device) for s in shapes]
    raw_args = [t.data for t in tensors]

    fwd_only  = time_raw(raw_fn, *raw_args, reps=reps_fwd)
    fwd_graph = time_fwd(op, *tensors, reps=reps_fwd)
    bwd_time  = time_bwd(op, *tensors, reps=reps_bwd)
    overhead  = fwd_graph - fwd_only

    return dict(op=name, device=device,
                fwd_only_ms=round(fwd_only, 4),
                fwd_graph_ms=round(fwd_graph, 4),
                overhead_ms=round(overhead, 4),
                bwd_ms=round(bwd_time, 4))

print("Timing helpers ready.")


In [ ]:
# ── Element-wise & activation op benchmarks ──────────────────────────────────
# Shape used: (512, 512) for element-wise, (1024, 1024) for activations

S2D   = (512, 512)
S2D_L = (1024, 1024)

OP_SPECS = [
    # (display_name, raw_backend_fn, ops_instance, [shapes])
    ("add",       lambda a, b: backend.add(a, b),         ops.add,       [S2D, S2D]),
    ("multiply",  lambda a, b: backend.multiply(a, b),    ops.multiply,  [S2D, S2D]),
    ("matmul",    lambda a, b: backend.matmul(a, b),      ops.matmul,    [S2D, S2D]),
    ("exp",       lambda a: backend.exp(a),               ops.exp,       [S2D]),
    ("log",       lambda a: backend.log(backend.abs(a) + 1e-6), ops.log, [S2D]),
    ("sqrt",      lambda a: backend.sqrt(backend.abs(a)), ops.sqrt,      [S2D]),
    ("sum",       lambda a: backend.sum(a),               ops.sum,       [S2D]),
    ("mean",      lambda a: backend.mean(a),              ops.mean,      [S2D]),
    ("relu",      lambda a: backend.maximum(a, 0),        ops.relu,      [S2D_L]),
    ("sigmoid",   lambda a: 1/(1+backend.exp(-a)),        ops.sigmoid,   [S2D_L]),
    ("tanh",      lambda a: backend.tanh(a),              ops.tanh,      [S2D_L]),
    ("softmax",   lambda a: backend.exp(a)/backend.sum(backend.exp(a),axis=-1,keepdims=True),
                            ops.softmax, [S2D_L]),
]

op_rows = []
for device in DEVICES:
    print(f"\n[{device.upper()}] benchmarking {len(OP_SPECS)} ops …")
    for name, raw_fn, op, shapes in OP_SPECS:
        row = benchmark_op(name, raw_fn, op, shapes, device=device)
        op_rows.append(row)
        print(f"  {name:12s}  fwd_only={row['fwd_only_ms']:.4f}ms  "
              f"fwd_graph={row['fwd_graph_ms']:.4f}ms  "
              f"overhead={row['overhead_ms']:.4f}ms  "
              f"bwd={row['bwd_ms']:.4f}ms")

df_ops = pd.DataFrame(op_rows)
print("\nDone.")


In [ ]:
# ── Conv2d & Pooling benchmarks ───────────────────────────────────────────────
# Input: (batch=16, C=32, H=32, W=32)  kernel 3×3
CONV_SHAPE  = (16, 32, 32, 32)
KERN_SHAPE  = (64, 32, 3, 3)

conv_rows = []
for device in DEVICES:
    print(f"\n[{device.upper()}] conv2d / pooling …")
    dnp.set_device(device)

    x_t = make_tensor(CONV_SHAPE, device)
    k_t = make_tensor(KERN_SHAPE, device)

    # conv2d
    fwd_c = time_fwd(ops.conv2d, x_t, k_t, reps=30)
    bwd_c = time_bwd(ops.conv2d, x_t, k_t, reps=10)
    print(f"  conv2d        fwd={fwd_c:.4f}ms  bwd={bwd_c:.4f}ms")
    conv_rows.append(dict(op="conv2d", device=device,
                          fwd_graph_ms=round(fwd_c,4), bwd_ms=round(bwd_c,4),
                          fwd_only_ms=float("nan"), overhead_ms=float("nan")))

    # max_pool2d
    x_p = make_tensor(CONV_SHAPE, device)
    fwd_p = time_fwd(ops.max_pool2d, x_p, reps=50)
    bwd_p = time_bwd(ops.max_pool2d, x_p, reps=20)
    print(f"  max_pool2d    fwd={fwd_p:.4f}ms  bwd={bwd_p:.4f}ms")
    conv_rows.append(dict(op="max_pool2d", device=device,
                          fwd_graph_ms=round(fwd_p,4), bwd_ms=round(bwd_p,4),
                          fwd_only_ms=float("nan"), overhead_ms=float("nan")))

    # avg_pool2d
    fwd_a = time_fwd(ops.avg_pool2d, x_p, reps=50)
    bwd_a = time_bwd(ops.avg_pool2d, x_p, reps=20)
    print(f"  avg_pool2d    fwd={fwd_a:.4f}ms  bwd={bwd_a:.4f}ms")
    conv_rows.append(dict(op="avg_pool2d", device=device,
                          fwd_graph_ms=round(fwd_a,4), bwd_ms=round(bwd_a,4),
                          fwd_only_ms=float("nan"), overhead_ms=float("nan")))

df_conv = pd.DataFrame(conv_rows)
print("\nDone.")


In [ ]:
# ── Section 1 Results Table & Plot ────────────────────────────────────────────
df_all_ops = pd.concat([df_ops, df_conv], ignore_index=True)

print("=" * 80)
print("Op Microbenchmark Results (ms)")
print("=" * 80)
print(df_all_ops.to_string(index=False))
print("=" * 80)

# --- Chart: forward latency comparison (CPU vs CUDA) -------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Op Microbenchmarks — Forward Latency (ms)", fontsize=14, fontweight="bold")

for ax, metric, title in zip(
    axes,
    ["fwd_graph_ms", "bwd_ms"],
    ["Forward (with autograd)", "Backward pass"],
):
    pivot = df_all_ops.pivot_table(index="op", columns="device", values=metric)
    pivot.plot(kind="bar", ax=ax, width=0.65, edgecolor="black")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("ms (log scale)")
    ax.set_yscale("log")
    ax.legend(title="device")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


---
## Section 2 — Device Tracking with `device_op_percentage`

`dnp.utils.device_op_percentage(device)` counts what fraction of all `Ops.__call__` invocations ran on a given device since the last `reset_device_stats()`.  
Useful for validating that a model is fully on GPU or diagnosing unintended CPU fallbacks.


In [ ]:
# ── Device tracking demo ──────────────────────────────────────────────────────
from dnp.core.session import session

def _run_mlp_forward(device, n_steps=20):
    """Run n_steps forward passes of a small MLP and return device %-stats."""
    reset_device_stats()
    dnp.set_device(device)

    W1 = make_tensor((128, 256), device)
    b1 = make_tensor((256,),     device)
    W2 = make_tensor((256, 128), device)
    b2 = make_tensor((128,),     device)
    W3 = make_tensor((128, 10),  device)
    b3 = make_tensor((10,),      device)

    for _ in range(n_steps):
        session.reset()
        x  = make_tensor((32, 128), device)
        h1 = ops.relu(ops.add(ops.matmul(x, W1), b1))
        h2 = ops.relu(ops.add(ops.matmul(h1, W2), b2))
        _  = ops.add(ops.matmul(h2, W3), b3)

    return {
        "device_requested": device,
        "cpu_%":  round(device_op_percentage("cpu"),  2),
        "cuda_%": round(device_op_percentage("cuda"), 2),
    }

tracking_rows = []
for dev in DEVICES:
    row = _run_mlp_forward(dev)
    tracking_rows.append(row)
    print(f"Requested={dev:4s}  →  CPU: {row['cpu_%']:6.2f}%   CUDA: {row['cuda_%']:6.2f}%")

# If CUDA available, show a mixed scenario (ops starting on CPU then switching)
if CUDA_OK:
    reset_device_stats()
    dnp.set_device("cpu")
    a = ops.add(make_tensor((256, 256), "cpu"), make_tensor((256, 256), "cpu"))
    dnp.set_device("cuda")
    b = ops.matmul(make_tensor((256, 256), "cuda"), make_tensor((256, 256), "cuda"))
    print(f"\nMixed scenario → CPU: {device_op_percentage('cpu'):.1f}%  "
          f"CUDA: {device_op_percentage('cuda'):.1f}%")


---
## Section 3 — Model Benchmarks

Three architectures are tested end-to-end:

| Model | Architecture | Task |
|-------|-------------|------|
| **MLP-S** | `128 → 256 → 128 → 10` | toy classification |
| **MLP-L** | `784 → 512 → 256 → 128 → 10` | larger MLP |
| **CNN** | `Conv2d(3→32,3×3) → ReLU → MaxPool → Conv2d(32→64,3×3) → ReLU → GAP → Linear(64,10)` | image-like input |

Metrics reported: **forward ms**, **backward ms**, and **params count**.


In [ ]:
# ── Model definitions ─────────────────────────────────────────────────────────

class MLP_S(dnp.Module):
    """Small MLP: 128 → 256 → 128 → 10."""
    def __init__(self):
        super().__init__()
        self.fc1 = dnp.Linear(128, 256, name="fc1")
        self.fc2 = dnp.Linear(256, 128, name="fc2")
        self.fc3 = dnp.Linear(128,  10, name="fc3")
    def forward(self, x):
        return self.fc3(ops.relu(self.fc2(ops.relu(self.fc1(x)))))


class MLP_L(dnp.Module):
    """Large MLP: 784 → 512 → 256 → 128 → 10."""
    def __init__(self):
        super().__init__()
        self.fc1 = dnp.Linear(784, 512, name="fc1")
        self.fc2 = dnp.Linear(512, 256, name="fc2")
        self.fc3 = dnp.Linear(256, 128, name="fc3")
        self.fc4 = dnp.Linear(128,  10, name="fc4")
    def forward(self, x):
        x = ops.relu(self.fc1(x))
        x = ops.relu(self.fc2(x))
        x = ops.relu(self.fc3(x))
        return self.fc4(x)


class CNN(dnp.Module):
    """Tiny CNN: 2 conv layers + global avg pool + linear head."""
    def __init__(self):
        super().__init__()
        self.conv1 = dnp.Conv2d(3,  32, kernel_size=3, padding=1, name="conv1")
        self.conv2 = dnp.Conv2d(32, 64, kernel_size=3, padding=1, name="conv2")
        self.pool  = dnp.MaxPool2d(kernel_size=2)
        self.gap   = dnp.GlobalAvgPool2d()
        self.fc    = dnp.Linear(64, 10, name="fc")
    def forward(self, x):
        x = ops.relu(self.conv1(x))
        x = self.pool(x)
        x = ops.relu(self.conv2(x))
        x = self.gap(x)
        return self.fc(x)


def count_params(model):
    return sum(p.data.size for p in model.parameters())

print("MLP-S  params:", count_params(MLP_S()))
print("MLP-L  params:", count_params(MLP_L()))
print("CNN    params:", count_params(CNN()))


In [ ]:
# ── Model forward + backward benchmark ───────────────────────────────────────
BATCH = 64       # samples per step
REPS  = 30       # repetitions

def _move_params(model, device):
    """Move all model parameters to the target device."""
    from dnp.core.backend import as_cupy, as_numpy
    for p in model.parameters():
        if device == "cuda":
            p.data = as_cupy(p.data)
        else:
            p.data = as_numpy(p.data)
        p.grad = p.grad * 0  # reset grad shape too

def benchmark_model(name, model_cls, batch_shape, device, reps=REPS):
    """Benchmark one model on one device. Returns a result dict."""
    dnp.set_device(device)
    model = model_cls()
    _move_params(model, device)
    loss_fn = dnp.CrossEntropyLoss()

    # Warmup
    for _ in range(3):
        session.reset()
        x = make_tensor(batch_shape, device)
        y_idx = np.random.randint(0, 10, size=(batch_shape[0],))
        from dnp.core.backend import as_cupy
        if device == "cuda":
            y_idx = as_cupy(y_idx)
        y = dnp.Tensor(y_idx)
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
    _sync()

    # Forward timing
    t_fwd = []
    for _ in range(reps):
        session.reset()
        x = make_tensor(batch_shape, device)
        _sync(); t0 = time.perf_counter()
        out = model(x)
        _sync()
        t_fwd.append((time.perf_counter() - t0) * 1000)

    # Forward+backward timing
    t_bwd = []
    for _ in range(reps):
        session.reset()
        x = make_tensor(batch_shape, device)
        y_idx = np.random.randint(0, 10, size=(batch_shape[0],))
        if device == "cuda":
            y_idx = as_cupy(y_idx)
        y = dnp.Tensor(y_idx)
        _sync(); t0 = time.perf_counter()
        out  = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        _sync()
        t_bwd.append((time.perf_counter() - t0) * 1000)

    return dict(
        model=name,
        device=device,
        params=count_params(model),
        fwd_ms      = round(float(np.mean(t_fwd)),  3),
        fwd_std_ms  = round(float(np.std(t_fwd)),   3),
        bwd_ms      = round(float(np.mean(t_bwd)),  3),
        bwd_std_ms  = round(float(np.std(t_bwd)),   3),
        throughput  = round(batch_shape[0] / (float(np.mean(t_bwd)) / 1000), 1),
    )


MODEL_SPECS = [
    ("MLP-S",  MLP_S, (BATCH, 128)),
    ("MLP-L",  MLP_L, (BATCH, 784)),
    ("CNN",    CNN,   (BATCH,  3, 32, 32)),
]

model_rows = []
for device in DEVICES:
    print(f"\n[{device.upper()}]")
    for mname, mcls, batch_shape in MODEL_SPECS:
        row = benchmark_model(mname, mcls, batch_shape, device)
        model_rows.append(row)
        print(f"  {mname:6s}  fwd={row['fwd_ms']:.3f}ms(±{row['fwd_std_ms']:.3f})  "
              f"bwd={row['bwd_ms']:.3f}ms(±{row['bwd_std_ms']:.3f})  "
              f"throughput={row['throughput']:.0f} samples/s")

df_models = pd.DataFrame(model_rows)


In [ ]:
# ── Batch-size scaling ────────────────────────────────────────────────────────
# How does MLP-L throughput scale with batch size on each device?

BATCH_SIZES = [8, 16, 32, 64, 128, 256]
SCALE_REPS  = 20

scale_rows = []
for device in DEVICES:
    print(f"\n[{device.upper()}] MLP-L batch-size scaling …")
    dnp.set_device(device)
    for bs in BATCH_SIZES:
        model = MLP_L()
        _move_params(model, device)
        # warmup
        for _ in range(3):
            session.reset()
            x = make_tensor((bs, 784), device)
            model(x)
        _sync()
        # measure
        times = []
        for _ in range(SCALE_REPS):
            session.reset()
            x = make_tensor((bs, 784), device)
            _sync(); t0 = time.perf_counter()
            model(x)
            _sync()
            times.append((time.perf_counter() - t0) * 1000)
        tp = round(bs / (float(np.mean(times)) / 1000), 1)
        scale_rows.append(dict(device=device, batch=bs,
                               fwd_ms=round(float(np.mean(times)), 3),
                               throughput=tp))
        print(f"  bs={bs:4d}  fwd={float(np.mean(times)):.3f}ms  throughput={tp:.0f} samples/s")

df_scale = pd.DataFrame(scale_rows)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("MLP-L — Batch-size Scaling", fontsize=13, fontweight="bold")

for device, grp in df_scale.groupby("device"):
    axes[0].plot(grp["batch"], grp["fwd_ms"],       marker="o", label=device)
    axes[1].plot(grp["batch"], grp["throughput"],   marker="o", label=device)

axes[0].set(title="Forward latency (ms)", xlabel="Batch size", ylabel="ms")
axes[1].set(title="Throughput (samples/s)", xlabel="Batch size", ylabel="samples/s")
for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## Section 4 — Training Throughput Benchmark

Run 50 epochs of a real training loop on synthetic data (10 k samples, 10-class).  
Reports **loss curve**, **epoch time**, and **samples/s**.


In [ ]:
# ── Training benchmark ────────────────────────────────────────────────────────
from dnp.utils import Trainer, EarlyStopping, History

N_SAMPLES  = 2000   # synthetic samples (keep small for speed)
N_FEATURES = 128
N_CLASSES  = 10
EPOCHS     = 30
BATCH_SIZE = 64

# Generate synthetic data (CPU numpy)
rng = np.random.default_rng(42)
X_np = rng.standard_normal((N_SAMPLES, N_FEATURES)).astype(np.float32)
y_np = rng.integers(0, N_CLASSES, size=N_SAMPLES)

train_results = {}

for device in DEVICES:
    print(f"\n[{device.upper()}] training MLP-S  {N_SAMPLES} samples × {EPOCHS} epochs …")
    reset_device_stats()
    dnp.set_device(device)

    model     = MLP_S()
    _move_params(model, device)
    optimizer = dnp.Adam(model.parameters(), lr=3e-3)
    loss_fn   = dnp.CrossEntropyLoss()

    history   = History()
    epoch_times = []

    for epoch in range(EPOCHS):
        t0 = time.perf_counter()
        session.reset()

        # Mini-batch SGD
        idx = rng.permutation(N_SAMPLES)
        losses_epoch = []
        for start in range(0, N_SAMPLES, BATCH_SIZE):
            batch_idx = idx[start:start + BATCH_SIZE]
            xb = X_np[batch_idx]
            yb = y_np[batch_idx]

            dnp.set_device(device)
            xb_t  = dnp.Tensor(xb)
            yb_t  = dnp.Tensor(yb)

            session.reset()
            optimizer.zero_grad()
            out  = model(xb_t)
            loss = loss_fn(out, yb_t)
            loss.backward()
            optimizer.step()
            losses_epoch.append(float(loss.data))

        epoch_ms     = (time.perf_counter() - t0) * 1000
        mean_loss    = float(np.mean(losses_epoch))
        throughput   = N_SAMPLES / (epoch_ms / 1000)
        epoch_times.append(epoch_ms)

        if (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1:3d}/{EPOCHS}  loss={mean_loss:.4f}  "
                  f"epoch_time={epoch_ms:.1f}ms  throughput={throughput:.0f} samp/s")
        history.history.setdefault("loss", []).append(mean_loss)

    train_results[device] = dict(
        history     = history,
        epoch_times = epoch_times,
        cpu_pct     = device_op_percentage("cpu"),
        cuda_pct    = device_op_percentage("cuda"),
    )
    print(f"  → Avg epoch: {np.mean(epoch_times):.1f}ms  "
          f"CPU%={device_op_percentage('cpu'):.1f}  CUDA%={device_op_percentage('cuda'):.1f}")


In [ ]:
# ── Training results plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Training Benchmark — MLP-S", fontsize=14, fontweight="bold")

colors = {"cpu": "#4C72B0", "cuda": "#DD8452"}

# Loss curves
ax = axes[0]
for dev, res in train_results.items():
    ax.plot(res["history"].history["loss"], label=dev, color=colors.get(dev))
ax.set(title="Loss curve", xlabel="Epoch", ylabel="Cross-entropy loss")
ax.legend(); ax.grid(True, alpha=0.3)

# Epoch time over training
ax = axes[1]
for dev, res in train_results.items():
    ax.plot(res["epoch_times"], label=dev, color=colors.get(dev), alpha=0.8)
ax.set(title="Epoch time (ms)", xlabel="Epoch", ylabel="ms")
ax.legend(); ax.grid(True, alpha=0.3)

# Device op % pie charts
ax = axes[2]
if len(train_results) == 1:
    dev  = list(train_results.keys())[0]
    res  = train_results[dev]
    vals = [res["cpu_pct"], res["cuda_pct"]]
    lbls = ["cpu", "cuda"]
    ax.pie(vals, labels=lbls, autopct="%1.1f%%",
           colors=[colors["cpu"], colors["cuda"]])
    ax.set_title(f"Device usage ({dev})")
else:
    # Grouped bars when both CPU and CUDA are available
    devs  = list(train_results.keys())
    cpus  = [train_results[d]["cpu_pct"]  for d in devs]
    cudas = [train_results[d]["cuda_pct"] for d in devs]
    x = np.arange(len(devs))
    ax.bar(x - 0.2, cpus,  0.4, label="cpu%",  color=colors["cpu"])
    ax.bar(x + 0.2, cudas, 0.4, label="cuda%", color=colors["cuda"])
    ax.set_xticks(x); ax.set_xticklabels(devs)
    ax.set(title="Device op %", ylabel="%")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout(); plt.show()


---
## Section 5 — GPU Memory Footprint  *(CUDA only)*

Measures peak CuPy memory allocated during model forward+backward — useful to estimate how large a batch can fit in VRAM.


In [ ]:
# ── GPU memory benchmark (CUDA only) ─────────────────────────────────────────
if not CUDA_OK:
    print("CUDA not available — skipping memory benchmark.")
else:
    import cupy as cp
    from dnp.core.backend import as_cupy

    def gpu_mem_mb():
        """Free, total GPU memory in MB for device 0."""
        pool  = cp.get_default_memory_pool()
        used  = pool.used_bytes()  / 1024**2
        total = pool.total_bytes() / 1024**2
        return used, total

    mem_rows = []
    dnp.set_device("cuda")
    loss_fn = dnp.CrossEntropyLoss()

    for mname, mcls, batch_shape in MODEL_SPECS:
        for bs in [16, 64, 128]:
            shape = (bs,) + batch_shape[1:]
            model = mcls()
            _move_params(model, "cuda")

            cp.get_default_memory_pool().free_all_blocks()
            synchronize()
            mem_before, _ = gpu_mem_mb()

            session.reset()
            x    = make_tensor(shape, "cuda")
            yidx = dnp.Tensor(as_cupy(np.random.randint(0, 10, bs)))
            out  = model(x)
            loss = loss_fn(out, yidx)
            loss.backward()
            synchronize()

            mem_after, total = gpu_mem_mb()
            delta = mem_after - mem_before

            mem_rows.append(dict(model=mname, batch=bs,
                                 mem_used_mb=round(mem_after, 2),
                                 mem_delta_mb=round(delta, 2)))
            print(f"  {mname:6s}  bs={bs:3d}  pool_used={mem_after:.2f}MB  delta={delta:.2f}MB")

    df_mem = pd.DataFrame(mem_rows)

    # Plot
    fig, ax = plt.subplots(figsize=(9, 4))
    for mname, grp in df_mem.groupby("model"):
        ax.plot(grp["batch"], grp["mem_delta_mb"], marker="o", label=mname)
    ax.set(title="GPU Memory Growth vs Batch Size (CUDA)",
           xlabel="Batch size", ylabel="CuPy pool delta (MB)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


---
## Section 6 — Summary Table

Combined results from all sections, formatted as a single reference table.


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
_sep = "─" * 65

print(f"\n{'='*65}")
print("  DifferentialNumpy v3 — Benchmark Summary")
print(f"{'='*65}")

print(f"\n{'OP MICROBENCHMARKS (fwd_graph, ms)':^65}")
print(_sep)
op_pivot = df_ops.pivot_table(index="op", columns="device", values="fwd_graph_ms")
print(op_pivot.to_string())

print(f"\n\n{'MODEL BENCHMARKS (batch={BATCH})':^65}".format(BATCH=BATCH))
print(_sep)
m_pivot = df_models[["model","device","params","fwd_ms","bwd_ms","throughput"]].copy()
print(m_pivot.to_string(index=False))

print(f"\n\n{'TRAINING THROUGHPUT':^65}")
print(_sep)
for dev, res in train_results.items():
    avg_epoch = float(np.mean(res["epoch_times"]))
    tp = N_SAMPLES / (avg_epoch / 1000)
    final_loss = res["history"].history["loss"][-1]
    print(f"  {dev:4s}  avg_epoch={avg_epoch:.1f}ms  "
          f"throughput={tp:.0f} samp/s  final_loss={final_loss:.4f}  "
          f"CPU%={res['cpu_pct']:.1f}  CUDA%={res['cuda_pct']:.1f}")

if CUDA_OK:
    print(f"\n\n{'GPU MEMORY (batch=64)':^65}")
    print(_sep)
    print(df_mem[df_mem["batch"] == 64][["model","mem_delta_mb"]].to_string(index=False))

print(f"\n{'='*65}")
print("  All benchmarks complete.")
print(f"{'='*65}\n")
